# Прогноз удержания пользователей

Цель — построить классическую ML-модель для прогноза `retention` и получить качественное ранжирование пользователей по метрике ROC-AUC.

Ноутбук показывает полный рабочий путь:

1. загрузка и проверка данных;
2. короткий EDA;
3. простые baseline-модели;
4. более сильные деревья и CatBoost;
5. осмысленное feature engineering;
6. ансамбль моделей;
7. mixture of experts по режимам активности;
8. обучение на всём train и создание submission.


## 1. Загрузка библиотек и данных

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from catboost import CatBoostClassifier
from scipy.stats import ks_2samp
from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sample_submission = pd.read_csv("data/sample_submission.csv")


**Комментарий.** Все основные настройки вынесены наверх. `sample_submission.csv` загружаем сразу, чтобы в финале точно сохранить правильные названия колонок.

In [ ]:
display(train.head())

display(pd.DataFrame({
    "dataset": ["train", "test", "sample_submission"],
    "rows": [len(train), len(test), len(sample_submission)],
    "columns": [train.shape[1], test.shape[1], sample_submission.shape[1]],
}))

print("Train columns:", train.columns.tolist())
print("Test columns: ", test.columns.tolist())


**Комментарий.** В train есть идентификатор, восемь рабочих признаков и целевая колонка `retention`. В test отсутствует только целевая переменная.

## 2. Проверка качества данных

In [ ]:
target_column = "retention"
feature_columns = [
    column for column in test.columns
    if column != "id"
]

quality_report = pd.DataFrame({
    "missing_train": train[feature_columns].isna().sum(),
    "missing_test": test[feature_columns].isna().sum(),
    "unique_train": train[feature_columns].nunique(),
    "unique_test": test[feature_columns].nunique(),
})

display(quality_report)
print("Infinite values in train:", np.isinf(train[feature_columns]).sum().sum())
print("Duplicate feature rows:", train.duplicated(feature_columns).sum())
print("Duplicate train IDs:", train["id"].duplicated().sum())
print("Duplicate test IDs:", test["id"].duplicated().sum())


**Комментарий.** Пропусков, бесконечных значений и дубликатов нет. Значит, сложная очистка или заполнение пропусков здесь не нужны.

In [ ]:
class_balance = train[target_column].value_counts().sort_index()

ax = class_balance.plot(
    kind="bar",
    color=["#7e57c2", "#ee8854"],
    figsize=(6, 4),
)
ax.set_title("Распределение целевого класса")
ax.set_xlabel("retention")
ax.set_ylabel("Количество объектов")
plt.xticks(rotation=0)
plt.show()

display(train[target_column].value_counts(normalize=True).rename("share"))


**Комментарий.** Классы несбалансированы умеренно: положительный класс занимает около 73%. Для сравнения моделей используем ROC-AUC и стратифицированную кросс-валидацию.

## 3. Короткий исследовательский анализ

In [ ]:
numeric_features = [
    column for column in feature_columns
    if column != "is_weekend_user"
]

train[numeric_features].hist(
    figsize=(14, 9),
    bins=30,
    color="#7e57c2",
)
plt.suptitle("Распределения числовых признаков", y=1.02)
plt.tight_layout()
plt.show()


**Комментарий.** Большинство признаков не имеют экстремально длинных хвостов. Логарифмирование и масштабирование для деревьев здесь не являются обязательными.

In [ ]:
corr = train.drop(columns="id").corr(numeric_only=True)

plt.figure(figsize=(11, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
)
plt.title("Корреляционная матрица")
plt.tight_layout()
plt.show()


**Комментарий.** Между `avg_session_time` и `avg_purchase_value` есть сильная связь. При этом линейные корреляции с target невысоки — вероятны нелинейные зависимости и взаимодействия.

In [ ]:
boundary_values = {
    "sessions_count": 1,
    "avg_session_time": 90,
    "avg_purchase_value": 10000,
    "active_days": 30,
    "session_std": 0,
}

boundary_report = []

for column, value in boundary_values.items():
    mask = train[column].eq(value)
    boundary_report.append({
        "feature": column,
        "boundary": value,
        "rows": mask.sum(),
        "share": mask.mean(),
        "retention_rate": train.loc[mask, target_column].mean(),
    })

display(pd.DataFrame(boundary_report))


**Комментарий.** Частые значения на границах похожи на clipping при генерации данных, а не на ошибки. Их не удаляем: например, `sessions_count = 1` описывает отдельную значимую группу пользователей.

In [ ]:
shift_rows = []

for column in feature_columns:
    ks_stat, p_value = ks_2samp(train[column], test[column])
    shift_rows.append({
        "feature": column,
        "train_mean": train[column].mean(),
        "test_mean": test[column].mean(),
        "KS": ks_stat,
        "p_value": p_value,
    })

shift_report = (
    pd.DataFrame(shift_rows)
    .sort_values("KS", ascending=False)
    .reset_index(drop=True)
)

display(shift_report)


**Комментарий.** Распределения train и test практически совпадают. Специальное перевзвешивание или удаление строк из-за dataset shift не требуется.

In [ ]:
regime_bins = [-np.inf, 1, 10, 16, np.inf]
regime_names = ["1", "2–10", "11–16", "17+"]

session_regime = pd.cut(
    train["sessions_count"],
    bins=regime_bins,
    labels=regime_names,
    include_lowest=True,
)

regime_table = (
    train.assign(session_regime=session_regime)
    .groupby("session_regime", observed=True)[target_column]
    .agg(objects="size", retention_rate="mean")
)

display(regime_table)


**Комментарий.** Поведение target различается между диапазонами `sessions_count`. Позже используем это наблюдение для mixture of experts — отдельных моделей для разных режимов активности.

## 4. Подготовка признаков и честная оценка

`id` исключаем: это технический идентификатор. Все преобразования создаём одинаково для train и test.

In [ ]:
X_base = train.drop(columns=["id", target_column]).copy()
X_test_base = test.drop(columns="id").copy()
y = train[target_column].copy()

# Random Forest получает простой индикатор граничного режима.
X_rf = X_base.copy()
X_test_rf = X_test_base.copy()
X_rf["is_one_session"] = X_rf["sessions_count"].eq(1).astype("int8")
X_test_rf["is_one_session"] = X_test_rf["sessions_count"].eq(1).astype("int8")

# CatBoost получает и числовые признаки, и категориальные копии счётчиков.
X_catcopy = X_base.copy()
X_test_catcopy = X_test_base.copy()
categorical_features = []

for column in [
    "sessions_count",
    "purchases_count",
    "active_days",
    "is_weekend_user",
]:
    new_column = f"{column}_cat"
    X_catcopy[new_column] = X_catcopy[column].astype(str)
    X_test_catcopy[new_column] = X_test_catcopy[column].astype(str)
    categorical_features.append(new_column)

print("Base shape:", X_base.shape)
print("CatBoost shape:", X_catcopy.shape)


**Комментарий.** Feature engineering минимальный и объяснимый: один бинарный индикатор для RF и категориальные копии дискретных счётчиков для CatBoost. PCA, кластеры, отношения и большие наборы ручных признаков в экспериментах не улучшили CV.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)


def make_oof(model, X, cv_splitter, fit_params=None):
    """Возвращает OOF-прогнозы и ROC-AUC каждого фолда."""
    fit_params = fit_params or {}
    oof = np.zeros(len(y))
    fold_scores = []

    for train_idx, val_idx in cv_splitter.split(X, y):
        fitted_model = clone(model)
        fitted_model.fit(
            X.iloc[train_idx],
            y.iloc[train_idx],
            **fit_params,
        )
        prediction = fitted_model.predict_proba(
            X.iloc[val_idx]
        )[:, 1]
        oof[val_idx] = prediction
        fold_scores.append(
            roc_auc_score(y.iloc[val_idx], prediction)
        )

    return oof, np.array(fold_scores)


model_results = []
oof_predictions = {}


def evaluate_model(name, model, X, fit_params=None):
    oof, scores = make_oof(model, X, cv, fit_params)
    oof_predictions[name] = oof
    model_results.append({
        "model": name,
        "OOF ROC-AUC": roc_auc_score(y, oof),
        "CV mean": scores.mean(),
        "CV std": scores.std(),
    })
    print(f"{name}: {roc_auc_score(y, oof):.6f}")


**Комментарий.** OOF-прогноз для каждой строки строится моделью, которая эту строку не видела. Это позволяет честно сравнивать модели и затем проверять blend без утечки.

## 5. Baseline-модели

In [ ]:
logistic = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE,
    ),
)

evaluate_model("Logistic Regression", logistic, X_base)


**Комментарий.** Логистическая регрессия — обязательный простой baseline. Ранее она давала около `0.59`, поэтому линейной границы для этой задачи недостаточно.

In [ ]:
decision_tree = DecisionTreeClassifier(
    max_depth=6,
    random_state=RANDOM_STATE,
)

evaluate_model("Decision Tree", decision_tree, X_base)


**Комментарий.** Неглубокое дерево уже улавливает нелинейности и заметно обгоняет логистическую регрессию, но одна модель остаётся нестабильной.

## 6. Более сильные классические модели

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=300,
    max_depth=11,
    min_samples_leaf=1,
    max_features=0.75,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

evaluate_model("Random Forest", random_forest, X_rf)


**Комментарий.** Random Forest усредняет много деревьев и поднимает качество примерно до `0.668`. Масштабирование признаков ему не требуется.

In [ ]:
gradient_boosting = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    random_state=RANDOM_STATE,
)

evaluate_model("Gradient Boosting", gradient_boosting, X_base)


**Комментарий.** Обычный Gradient Boosting сильнее baseline, но в этой задаче уступает случайному лесу и CatBoost.

In [ ]:
catboost_numeric = CatBoostClassifier(
    iterations=500,
    depth=4,
    learning_rate=0.03,
    l2_leaf_reg=3,
    random_strength=1,
    random_seed=RANDOM_STATE,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=False,
    allow_writing_files=False,
)

evaluate_model("CatBoost numeric", catboost_numeric, X_base)


**Комментарий.** CatBoost хорошо моделирует сложные взаимодействия. Около 500 итераций оказалось лучше 1000: более длинное обучение начинало переобучаться.

In [ ]:
catboost_catcopy = CatBoostClassifier(
    iterations=500,
    depth=4,
    learning_rate=0.03,
    l2_leaf_reg=3,
    random_strength=1,
    random_seed=RANDOM_STATE,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=False,
    allow_writing_files=False,
)

evaluate_model(
    "CatBoost + cat copies",
    catboost_catcopy,
    X_catcopy,
    fit_params={"cat_features": categorical_features},
)


**Комментарий.** Категориальные копии позволяют CatBoost отдельно моделировать ступенчатые эффекты счётчиков. Этот вариант особенно полезен внутри ансамбля.

In [ ]:
extra_trees = ExtraTreesClassifier(
    n_estimators=800,
    max_depth=11,
    min_samples_leaf=5,
    max_features=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

evaluate_model("ExtraTrees", extra_trees, X_base)


**Комментарий.** ExtraTrees немного слабее лучшего CatBoost отдельно, но строит другие границы и поэтому даёт полезное разнообразие для blend.

In [ ]:
comparison = (
    pd.DataFrame(model_results)
    .sort_values("OOF ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(comparison)


**Комментарий.** Сравниваем модели только по одинаковой CV-схеме. Слабая отдельная модель всё ещё может улучшать ансамбль, если её ошибки отличаются от ошибок лидера.

## 7. Blend глобальных моделей

In [ ]:
cat_oof = oof_predictions["CatBoost + cat copies"]
rf_oof = oof_predictions["Random Forest"]
extra_oof = oof_predictions["ExtraTrees"]

prediction_corr = pd.DataFrame({
    "CatBoost": cat_oof,
    "Random Forest": rf_oof,
    "ExtraTrees": extra_oof,
}).corr()

display(prediction_corr)

weight_grid = [
    (0.30, 0.30, 0.40),
    (0.40, 0.20, 0.40),
    (0.40, 0.25, 0.35),
    (0.45, 0.20, 0.35),
    (0.45, 0.25, 0.30),
    (0.50, 0.20, 0.30),
]

blend_rows = []

for weights in weight_grid:
    w_cat, w_rf, w_extra = weights
    prediction = (
        w_cat * cat_oof
        + w_rf * rf_oof
        + w_extra * extra_oof
    )
    blend_rows.append({
        "cat_weight": w_cat,
        "rf_weight": w_rf,
        "extra_weight": w_extra,
        "ROC-AUC": roc_auc_score(y, prediction),
    })

blend_table = (
    pd.DataFrame(blend_rows)
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(blend_table)

selected_global_weights = tuple(
    blend_table.loc[
        0,
        ["cat_weight", "rf_weight", "extra_weight"],
    ]
)


**Комментарий.** Вероятности усредняем, а не округляем до классов. Лучшим и устойчивым сочетанием в экспериментах стало примерно `50% CatBoost + 20% RF + 30% ExtraTrees`.

## 8. Mixture of experts

Одна глобальная модель должна описать всех пользователей сразу. Попробуем отдельные CatBoost-модели для четырёх режимов `sessions_count`.

In [ ]:
regime_labels = pd.cut(
    X_base["sessions_count"],
    bins=regime_bins,
    labels=False,
    include_lowest=True,
).astype(int)


def make_regime_oof(cv_splitter):
    oof = np.zeros(len(y))

    for train_idx, val_idx in cv_splitter.split(X_base, y):
        train_regimes = regime_labels.iloc[train_idx]
        val_regimes = regime_labels.iloc[val_idx]

        for regime in sorted(regime_labels.unique()):
            train_mask = train_regimes.eq(regime).to_numpy()
            val_mask = val_regimes.eq(regime).to_numpy()

            model = CatBoostClassifier(
                iterations=300,
                depth=3,
                learning_rate=0.03,
                l2_leaf_reg=7,
                random_strength=1,
                random_seed=RANDOM_STATE,
                loss_function="Logloss",
                verbose=False,
                allow_writing_files=False,
            )
            model.fit(
                X_base.iloc[train_idx].loc[train_mask],
                y.iloc[train_idx].loc[train_mask],
            )

            regime_val_idx = val_idx[val_mask]
            oof[regime_val_idx] = model.predict_proba(
                X_base.iloc[regime_val_idx]
            )[:, 1]

    return oof


regime_oof = make_regime_oof(cv)
print("Regime OOF ROC-AUC:", roc_auc_score(y, regime_oof))


**Комментарий.** Четыре специализированных эксперта дают качество на уровне сильного ансамбля. Их прогнозы полезно смешать с глобальной моделью, которая лучше согласует пользователей из разных режимов.

In [ ]:
seeds = [42, 123, 777, 2026, 3407]
regime_weight_grid = np.arange(0.0, 0.41, 0.05)
weight_scores = {
    float(weight): [] for weight in regime_weight_grid
}

w_cat, w_rf, w_extra = selected_global_weights

for seed in seeds:
    cv_seed = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed,
    )

    cat_seed, _ = make_oof(
        catboost_catcopy,
        X_catcopy,
        cv_seed,
        {"cat_features": categorical_features},
    )
    rf_seed, _ = make_oof(random_forest, X_rf, cv_seed)
    extra_seed, _ = make_oof(extra_trees, X_base, cv_seed)
    regime_seed = make_regime_oof(cv_seed)

    global_seed = (
        w_cat * cat_seed
        + w_rf * rf_seed
        + w_extra * extra_seed
    )

    for weight in regime_weight_grid:
        weight = float(weight)
        prediction = (
            (1.0 - weight) * global_seed
            + weight * regime_seed
        )
        weight_scores[weight].append(
            roc_auc_score(y, prediction)
        )

regime_weight_table = pd.DataFrame([
    {
        "regime_weight": weight,
        "global_weight": 1.0 - weight,
        "mean_auc": np.mean(scores),
        "std_auc": np.std(scores),
        "min_auc": np.min(scores),
    }
    for weight, scores in weight_scores.items()
]).sort_values(
    ["mean_auc", "std_auc"],
    ascending=[False, True],
).reset_index(drop=True)

display(regime_weight_table)

best_regime_weight = float(
    regime_weight_table.loc[0, "regime_weight"]
)
print("Selected regime weight:", best_regime_weight)


**Комментарий.** Финальный вес выбирается не по одному удачному разбиению, а по среднему результату пяти разных CV seeds. Эта ячейка самая долгая, зато защищает от случайного выбора blend.

## 9. Обучение на всём train и submission

In [ ]:
# Обучаем финальные модели на всём train.
cat_final = clone(catboost_catcopy)
cat_final.fit(X_catcopy, y, cat_features=categorical_features)
pred_cat = cat_final.predict_proba(X_test_catcopy)[:, 1]

rf_final = clone(random_forest)
rf_final.fit(X_rf, y)
pred_rf = rf_final.predict_proba(X_test_rf)[:, 1]

extra_final = clone(extra_trees)
extra_final.fit(X_base, y)
pred_extra = extra_final.predict_proba(X_test_base)[:, 1]

# Финальные веса зафиксированы по repeated OOF CV.
pred_final = 0.50 * pred_cat + 0.20 * pred_rf + 0.30 * pred_extra

submission_target = next(column for column in sample_submission.columns if column != "id")
submission = pd.DataFrame({"id": test["id"], submission_target: pred_final})

assert submission.shape == sample_submission.shape
assert submission[submission_target].between(0, 1).all()

output_path = "results/submission_final.csv"
submission.to_csv(output_path, index=False)

display(submission.head())
print("Saved:", output_path)
print("Shape:", submission.shape)
print("Prediction range:", pred_final.min(), pred_final.max())


**Комментарий.** Финальные модели обучены на всех размеченных данных. В submission записаны вероятности класса `1`, а не классы `0/1`, потому что ROC-AUC оценивает качество ранжирования.